In [5]:
import logging
import logging.handlers
import json
import difflib
import httpx
import time
from typing import List, Dict, Optional, Tuple
from urllib.parse import urljoin
import pandas as pd
from difflib import SequenceMatcher
from bs4 import BeautifulSoup
from urllib.parse import urlparse

from config import Config

# Logging setup with file rotation
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
file_handler = logging.handlers.RotatingFileHandler(
    'plant_info.log', maxBytes=1_000_000, backupCount=3
)
file_handler.setFormatter(logging.Formatter(
    '%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    '%Y-%m-%d %H:%M:%S'
))
logging.getLogger().addHandler(file_handler)

logger = logging.getLogger(__name__)

def redact_sensitive(text: str) -> str:
    return text.replace(Config.PLANTNET_KEY, "<api-key>")


class PlantInfoFetcher:
    """Helper for identifying plants and retrieving care information."""

    def __init__(self, timeout: float = 10.0) -> None:
        self.timeout = timeout
        logger.info("Initialized PlantInfoFetcher with timeout=%s", timeout)

    async def identify_with_plantnet(
        self,
        files: List[tuple[str, str]],
        data: Dict = None,
    ) -> Dict[str, object]:
        api_url = f"{Config.PLANTNET_API}{Config.PROJECT}?api-key={Config.PLANTNET_KEY}"
        safe_url = redact_sensitive(api_url)
        logger.debug("Sending image data to PlantNet API at %s", safe_url)
        try:
            start = time.monotonic()
            async with httpx.AsyncClient(timeout=self.timeout) as client:
                response = await client.post(api_url, data=data, files=files)
            elapsed = time.monotonic() - start
            logger.info("PlantNet API call took %.2f seconds", elapsed)
            response.raise_for_status()
            payload = response.json()
            logger.debug("PlantNet response: %s", json.dumps(payload)[:500])
            results = payload.get("results", [])
            if not results:
                logger.warning("No results from PlantNet API")
                raise ValueError("No results returned from PlantNet API")
            top = results[0]
            species = top.get("species", {})
            return {
                "score": top.get("score"),
                "common_names": species.get("commonNames", []),
                "scientific_name": species.get("scientificName"),
            }
        except httpx.RequestError as e:
            logger.exception("Request to PlantNet API failed")
            raise
        except (KeyError, ValueError, TypeError) as e:
            logger.error("Error parsing PlantNet response: %s", e)
            raise

    
    def _rank_items_by_similarity(self, items, target):
        """
        Casts all items to str and computes SequenceMatcher ratio,
        treating NaNs or other types as empty strings.
        """
        # Replace NaN with empty string, ensure string type
        clean_items = items.fillna("").astype(str)
        return clean_items.apply(lambda x: SequenceMatcher(None, x, target).ratio())


    def _get_best_match_row(self, df, column, target):
        """
        Finds the row in DataFrame `df` whose `column` string is most similar to `target`.

        Args:
            df (pd.DataFrame): The DataFrame containing the data.
            column (str): The name of the column with strings to compare.
            target (str): The reference string to match against.

        Returns:
            pd.Series: The row from `df` with the highest similarity score.
        """
        # Calculate similarity scores
        scores = self._rank_items_by_similarity(df[column], target)
        # Identify the index of the max score
        best_idx = scores.idxmax()
        # Optionally, attach the score to the result for inspection
        best_row = df.loc[best_idx].copy()
        best_row['similarity_score'] = scores[best_idx]
        return dict(best_row)

            
    def _parse_perenual_api_data(self, data: Dict) -> pd.DataFrame:
        """
        Parses the care data from the provided JSON structure into a pandas DataFrame.
        
        Args:
            data (Dict): The JSON data containing plant care information.
        
        Returns:
            pd.DataFrame: A DataFrame with columns for scientific name, common names, watering, sunlight, and pruning.
        """
        if 'data' not in data or not isinstance(data['data'], list):
            raise ValueError("Invalid data format: 'data' key missing or not a list")
        else:
            # Prepare rows for the DataFrame
            rows = []
            for item in data['data']:
                species_id = item.get('species_id', None)
                sci_names = item.get('scientific_name', None)
                scientific_name = sci_names[0] if isinstance(sci_names, list) else "None"
                common_name = item.get('common_name', None)

                # Extract section descriptions by type
                watering = sunlight = pruning = None
                for section in item.get('section', []):
                    t = section.get('type')
                    desc = section.get('description')
                    if t == 'watering':
                        watering = desc
                    elif t == 'sunlight':
                        sunlight = desc
                    elif t == 'pruning':
                        pruning = desc

                rows.append({
                    'species id': species_id,
                    'perenual scientific name': scientific_name,
                    'perenual common name': common_name,
                    'watering': watering,
                    'sunlight': sunlight,
                    'pruning': pruning
                })

            # Create the DataFrame
            df = pd.DataFrame(rows, columns=['species id', 'perenual scientific name', 'perenual common name', 'watering', 'sunlight', 'pruning'])

            return df


    async def fetch_care_from_perenual(self, plant_name: str) -> pd.DataFrame:
        """Fetch care data via API, but on 429 rate limit, fall back to scraping."""
        logger.info("Fetching care data for plant: %s", plant_name)
        url = (
            f"{Config.PERENUAL_API}{Config.PERENUAL_ENDPOINT}?key={Config.PERENUAL_KEY}&q={plant_name}"
        )
        safe_url = redact_sensitive(url)
        logger.debug("Calling URL: %s", safe_url)
        try:
            start = time.monotonic()
            async with httpx.AsyncClient(timeout=self.timeout) as client:
                resp = await client.get(url)
            elapsed = time.monotonic() - start
            logger.info("Perenual API call took %.2f seconds", elapsed)
            resp.raise_for_status()
            data = resp.json()
            logger.debug("Fetched care data: %s", json.dumps(data)[:500])
            return self._parse_perenual_api_data(data)

        except httpx.HTTPStatusError as exc:
            if exc.response.status_code == 429:
                logger.warning(
                    "Perenual API quota exceeded for %s. Falling back to scraping.", plant_name
                )
                # Fallback: scrape the website directly
                scraped = []
                search_urls = await self.scrape_perenual_search(plant_name)
                for page_url in search_urls:
                    care_info = await self.scrape_perenual_page(page_url)
                    care_info['search_term'] = plant_name
                    scraped.append(care_info)
                # Build DataFrame from scraped entries
                return pd.DataFrame(
                    scraped,
                    columns=[
                        'search_term', 'species id', 'perenual scientific name',
                        'perenual common name', 'watering', 'sunlight', 'pruning'
                    ]
                )
            else:
                logger.error("Failed fetching care guide for %s: %s", plant_name, exc)
                return pd.DataFrame(
                    columns=[
                        'search_term', 'species id', 'perenual scientific name',
                        'perenual common name', 'watering', 'sunlight', 'pruning'
                    ]
                )
        except (httpx.HTTPError, json.JSONDecodeError) as exc:
            logger.error("Error fetching care guide for %s: %s", plant_name, exc)
            return pd.DataFrame(
                columns=[
                    'search_term', 'species id', 'perenual scientific name',
                    'perenual common name', 'watering', 'sunlight', 'pruning'
                ]
            )


    async def scrape_perenual_search(self, query: str) -> List[Tuple[str, str]]:
        logger.info("Scraping search results for: %s", query)
        search_url = "https://perenual.com/plant-species-database-search-finder"
        base = "https://perenual.com/plant-database-search-guide/species/"
        headers = {"User-Agent": "Mozilla/5.0"}
        try:
            start = time.monotonic()
            async with httpx.AsyncClient(timeout=self.timeout) as client:
                resp = await client.get(search_url, params={"search": query}, headers=headers)
            elapsed = time.monotonic() - start
            logger.info("Search page scrape took %.2f seconds", elapsed)
            resp.raise_for_status()
            logger.debug("Received HTML response for search query")
        except httpx.HTTPError as exc:
            logger.error("Error searching Perenual: %s", exc)
            return []
        soup = BeautifulSoup(resp.text, "html.parser")
        results: List[Tuple[str, str]] = []
        for a in soup.select(".search-container-box"):
            url = urljoin(base, a.get("href"))
            results.append(url)
        logger.debug("Extracted %d search results", len(results))
        return results


    def _extract_id_with_urlparse(self, url: str) -> str:
        path = urlparse(url).path      # e.g. '/plant-species-database-search-finder/species/3379'
        return path.rstrip('/').split('/')[-1]


    async def scrape_perenual_page(self, url: str) -> Dict[str, str]:
        logger.info("Scraping care page at: %s", url)
        try:
            start = time.monotonic()
            async with httpx.AsyncClient(timeout=self.timeout) as client:
                resp = await client.get(url)
            elapsed = time.monotonic() - start
            logger.info("Care page scrape took %.2f seconds", elapsed)
            resp.raise_for_status()
            logger.debug("Received HTML content for care page")
        except httpx.HTTPError as exc:
            logger.error("Failed to fetch Perenual page: %s", exc)
            return {}

        soup = BeautifulSoup(resp.text, "html.parser")
        
        paras = soup.select("p.whitespace-pre-wrap")
        texts = [p.get_text(strip=True) for p in paras[:3]]
        keys = ["watering", "sunlight", "pruning"]
        result = dict(zip(keys, texts))
        logger.debug("Extracted care information: %s", result)

        result["species id"] = self._extract_id_with_urlparse(url)

        # Extract scientific name and common names
        common_name = soup.select("h1")
        logger.info("Extracted Common Name: %s", common_name[0].get_text(strip=True))
        result['perenual common name'] = common_name[0].get_text(strip=True) if common_name else "Unknown"

        scientific_name = soup.select("h2")
        logger.info("Extracted Scientific Name: %s", scientific_name[0].get_text(strip=True))
        result['perenual scientific name'] = scientific_name[0].get_text(strip=True) if scientific_name else "Unknown"

        logger.debug("Final care information: %s", result)
        
        return result


    async def get_plant_info(self, image_files: list, organs: list) -> Dict[str, object]:
        logger.info("Identifying plant and retrieving care information")
        files = []
        for image_file in image_files:
            logger.debug("Reading image file: %s", image_file)
            image_data = open(image_file, 'rb')
            files.append(('images', image_data))

        if organs is None:
            organs = ["auto" for _ in image_files]
        logger.debug("Using organs list: %s", organs)

        data = {'organs': organs}
        logger.info("Sending identification request with data: %s", data)

        identified = await self.identify_with_plantnet(data=data, files=files)

        names = [identified.get("scientific_name", "")] + identified.get("common_names", [])                    
        df = pd.DataFrame(columns=['search_term', 'species id', 'perenual scientific name', 'perenual common name', 'watering', 'sunlight', 'pruning'])

        logger.debug("Attempting to fetch care data for names: %s", names)
        for name in names:
            care_df = await self.fetch_care_from_perenual(name)
            care_df['search_term'] = name
            df = pd.concat([df, care_df], ignore_index=True)
            logger.info("Number of care entries found for '%s': %d", name, len(care_df))
            logger.info("Total number of entries: %s ", len(df))
        
        best_match = self._get_best_match_row(df, "perenual scientific name", identified.get("scientific_name", ""))
        if best_match['similarity_score'] < 0.5:
            logger.warning("No sufficiently similar match found for scientific name '%s'", identified.get("scientific_name", ""))
            scraped_care_array = []
            for name in names:
                search_results = await self.scrape_perenual_search(name)
                
                for _url in search_results:
                    care_dict = await self.scrape_perenual_page(_url)
                    care_dict['search_term'] = name
                    scraped_care_array.append(care_dict)
                
            scraped_care_df = pd.DataFrame(scraped_care_array, columns=['search_term', 'species id', 'perenual scientific name', 'perenual common name', 'watering', 'sunlight', 'pruning'])
            
            best_match = self._get_best_match_row(scraped_care_df, "perenual scientific name", identified.get("scientific_name", ""))
            df = pd.concat([df, scraped_care_df], ignore_index=True)

        identified.update(best_match)
        logger.info("Completed identification and care retrieval")
        return identified, df




In [9]:
fetcher = PlantInfoFetcher()
image_files = ["./../../plant_id_test/test_photos/camellia.jpg"]
organs = ["auto"]
result, all_df = await fetcher.get_plant_info(image_files=image_files, organs=organs)
result

2025-07-26 23:32:47 [INFO] __main__: Initialized PlantInfoFetcher with timeout=10.0
2025-07-26 23:32:47 [INFO] __main__: Identifying plant and retrieving care information
2025-07-26 23:32:47 [INFO] __main__: Sending identification request with data: {'organs': ['auto']}
2025-07-26 23:32:49 [INFO] httpx: HTTP Request: POST https://my-api.plantnet.org/v2/identify/all?api-key=2b10uztmWSXBQhAsctxZTu "HTTP/1.1 200 OK"
2025-07-26 23:32:49 [INFO] __main__: PlantNet API call took 2.27 seconds
2025-07-26 23:32:49 [INFO] __main__: Fetching care data for plant: Camellia sasanqua Thunb.
2025-07-26 23:32:50 [INFO] httpx: HTTP Request: GET https://perenual.com/api/species-care-guide-list?key=sk-BY6T687a9e958639511466&q=Camellia%20sasanqua%20Thunb. "HTTP/1.1 200 OK"
2025-07-26 23:32:50 [INFO] __main__: Perenual API call took 0.83 seconds
2025-07-26 23:32:50 [INFO] __main__: Number of care entries found for 'Camellia sasanqua Thunb.': 0
2025-07-26 23:32:50 [INFO] __main__: Total number of entries: 0 


{'score': 0.36081,
 'common_names': ['Camellia', 'Sasanqua camellia', 'Christmas camellia'],
 'scientific_name': 'Camellia sasanqua Thunb.',
 'search_term': 'Camellia',
 'species id': 1538,
 'perenual scientific name': 'Camellia sasanqua',
 'perenual common name': 'camellia',
 'watering': 'Camellia sasanqua requires regular watering during the summer months. Generally, this plant should be watered every 1 to 2 weeks for optimal growth. In warmer climates, it may need extra water during the warmer months, but it should not be watered every day.\n\nIn cooler temperatures, Camellia sasanqua should be watered once every 2 to 3 weeks. During this time, the soil should be kept moist, but not soggy. If the soil is dry approximately 1 inch below the surface, it’s time to water.\n\nIn periods of extreme heat, Camellia sasanqua needs to be watered more frequently to ensure proper hydration and to prevent burning of the foliage. During these times, you should water the plant every week to ensure 

In [ ]:
fetcher = PlantInfoFetcher()
image_files = ["./../../plant_id_test/test_photos/lavender.jpg"]
organs = ["auto"]
result, all_df = await fetcher.get_plant_info(image_files=image_files, organs=organs)
result

2025-07-26 23:53:07 [INFO] __main__: Initialized PlantInfoFetcher with timeout=10.0
2025-07-26 23:53:07 [INFO] __main__: Identifying plant and retrieving care information
2025-07-26 23:53:07 [INFO] __main__: Sending identification request with data: {'organs': ['auto']}
2025-07-26 23:53:10 [INFO] httpx: HTTP Request: POST https://my-api.plantnet.org/v2/identify/all?api-key=2b10uztmWSXBQhAsctxZTu "HTTP/1.1 200 OK"
2025-07-26 23:53:10 [INFO] __main__: PlantNet API call took 2.95 seconds
2025-07-26 23:53:10 [INFO] __main__: Fetching care data for plant: Lavandula angustifolia Mill.
2025-07-26 23:53:11 [INFO] httpx: HTTP Request: GET https://perenual.com/api/species-care-guide-list?key=sk-Qe3y68855a240f67111574&q=Lavandula%20angustifolia%20Mill. "HTTP/1.1 429 Too Many Requests"
2025-07-26 23:53:11 [INFO] __main__: Perenual API call took 1.28 seconds
2025-07-26 23:53:11 [WARNING] __main__: Perenual API quota exceeded for Lavandula angustifolia Mill.. Falling back to scraping.
2025-07-26 23:

{'score': 0.73563,
 'common_names': ['English lavender', 'Common Lavender', 'Lavender'],
 'scientific_name': 'Lavandula angustifolia Mill.',
 'search_term': 'English lavender',
 'species id': '4702',
 'perenual scientific name': 'Lavandula angustifolia',
 'perenual common name': 'English lavender',
 'watering': 'English lavender (Lavandula angustifolia) should be watered deeply, but infrequently. They should be given approximately 1.5-2 inches of water a week, but only when the soil is dry to the touch. This can be checked by sticking your finger into the soil up to your first knuckle; if the soil is dry, it is time to water. If the soil is still moist, hold off on watering for a few days. As the temperatures begin to become warm, English lavender should be watered once every 3-4 days. If the temperatures become extreme, avoid watering the plant during the hottest part of the day (i.e., noon through 4 pm). The best time to water is in the early morning or evening when it is cooler. Avo

In [3]:
fetcher = PlantInfoFetcher()
image_files = ["./../../plant_id_test/test_photos/mint.jpg"]
organs = ["auto"]
result, all_df = await fetcher.get_plant_info(image_files=image_files, organs=organs)
result

2025-07-26 23:46:53 [INFO] __main__: Initialized PlantInfoFetcher with timeout=10.0
2025-07-26 23:46:53 [INFO] __main__: Identifying plant and retrieving care information
2025-07-26 23:46:53 [INFO] __main__: Sending identification request with data: {'organs': ['auto']}
2025-07-26 23:46:55 [INFO] httpx: HTTP Request: POST https://my-api.plantnet.org/v2/identify/all?api-key=2b10uztmWSXBQhAsctxZTu "HTTP/1.1 200 OK"
2025-07-26 23:46:55 [INFO] __main__: PlantNet API call took 1.88 seconds
2025-07-26 23:46:55 [INFO] __main__: Fetching care data for plant: Mentha spicata L.
2025-07-26 23:46:56 [INFO] httpx: HTTP Request: GET https://perenual.com/api/species-care-guide-list?key=sk-Qe3y68855a240f67111574&q=Mentha%20spicata%20L. "HTTP/1.1 429 Too Many Requests"
2025-07-26 23:46:56 [INFO] __main__: Perenual API call took 0.81 seconds
2025-07-26 23:46:56 [ERROR] __main__: Failed fetching care guide for Mentha spicata L.: Client error '429 Too Many Requests' for url 'https://perenual.com/api/speci

TypeError: cannot concatenate object of type '<class 'dict'>'; only Series and DataFrame objs are valid

In [4]:
fetcher = PlantInfoFetcher()
image_files = ["./../../plant_id_test/test_photos/rockrose.jpg"]
organs = ["auto"]
result, all_df = await fetcher.get_plant_info(image_files=image_files, organs=organs)
result

2025-07-26 23:47:04 [INFO] __main__: Initialized PlantInfoFetcher with timeout=10.0
2025-07-26 23:47:04 [INFO] __main__: Identifying plant and retrieving care information
2025-07-26 23:47:04 [INFO] __main__: Sending identification request with data: {'organs': ['auto']}
2025-07-26 23:47:04 [INFO] httpx: HTTP Request: POST https://my-api.plantnet.org/v2/identify/all?api-key=2b10uztmWSXBQhAsctxZTu "HTTP/1.1 200 OK"
2025-07-26 23:47:04 [INFO] __main__: PlantNet API call took 0.66 seconds
2025-07-26 23:47:04 [INFO] __main__: Fetching care data for plant: Cistus crispus L.
2025-07-26 23:47:05 [INFO] httpx: HTTP Request: GET https://perenual.com/api/species-care-guide-list?key=sk-Qe3y68855a240f67111574&q=Cistus%20crispus%20L. "HTTP/1.1 429 Too Many Requests"
2025-07-26 23:47:05 [INFO] __main__: Perenual API call took 0.73 seconds
2025-07-26 23:47:05 [ERROR] __main__: Failed fetching care guide for Cistus crispus L.: Client error '429 Too Many Requests' for url 'https://perenual.com/api/speci

TypeError: cannot concatenate object of type '<class 'dict'>'; only Series and DataFrame objs are valid